### 01 — Data Cleaning, Transformation & Feature Engineering

Explores raw dataset, applies documented `queue -> category` mapping, clean text, handles missing/near-empty data, engineers inference-safe numeric features, and produces `data/clean_tickets.csv`

**Dataset**: 
Multilingual Customer Support Tickets (Tobias Bueck, HuggingFace).

**Link**: https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets/blob/main/dataset-tickets-multi-lang-4-20k.csv

**Disclosure:** this dataset is synthetically
generated per its own documentation, not scraped real-world tickets

In [ ]:
import re
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, '..')
from app.features import FEATURE_NAMES, extract_numeric_features

pd.set_option('display.max_colwidth',120)

RAW_PATH = Path("../data/raw_tickets.csv")
CLEAN_PATH = Path("../data/clean_tickets.csv")

1. Load raw data and inspect

In [2]:
df = pd.read_csv(RAW_PATH)
df.head(3)

,subject,body,answer,type,queue,priority,language,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Unvorhergesehener Absturz der Datenanalyse-Plattform,"Die Datenanalyse-Plattform brach unerwartet ab, da die Speicheroberfläche zu gering war. Ich habe versucht, Laravel ...","Ich werde Ihnen bei der Lösung des Problems helfen, indem die Datenanalyse-Plattform neu gestartet wird. Bitte berei...",Incident,General Inquiry,low,de,Crash,Technical,Bug,Hardware,Resolution,Outage,Documentation,NaN
1,Customer Support Inquiry,Seeking information on digital strategies that can aid in brand growth and details on the available services. Lookin...,"We offer a variety of digital strategies and services to boost brand growth, including social media management and m...",Request,Customer Service,medium,en,Feedback,Sales,IT,Tech Support,NaN,NaN,NaN,NaN
2,Data Analytics for Investment,I am contacting you to request information on data analytics tools that can be utilized with the Eclipse IDE for enh...,I am here to assist you with data analytics tools for investment optimization. Please provide more details on your r...,Request,Customer Service,medium,en,Technical,Product,Guidance,Documentation,Performance,Feature,NaN,NaN


In [3]:
df.shape

(20000, 15)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   18539 non-null  object
 1   body      19998 non-null  object
 2   answer    19996 non-null  object
 3   type      20000 non-null  object
 4   queue     20000 non-null  object
 5   priority  20000 non-null  object
 6   language  20000 non-null  object
 7   tag_1     20000 non-null  object
 8   tag_2     19954 non-null  object
 9   tag_3     19905 non-null  object
 10  tag_4     18461 non-null  object
 11  tag_5     13091 non-null  object
 12  tag_6     7351 non-null   object
 13  tag_7     3928 non-null   object
 14  tag_8     1907 non-null   object
dtypes: object(15)
memory usage: 2.3+ MB


In [5]:
df.isnull().sum()

subject      1461
body            2
answer          4
type            0
queue           0
priority        0
language        0
tag_1           0
tag_2          46
tag_3          95
tag_4        1539
tag_5        6909
tag_6       12649
tag_7       16072
tag_8       18093
dtype: int64

2. Language filter (only english)

In [6]:
df_en = df[df['language'] == 'en'].copy()
df_en.shape

(11923, 15)

3. duplicate check

In [7]:
df_en.duplicated().sum()

0

In [8]:
df_en.duplicated(subset=['subject', 'body']).sum()

0

4. `tag_1`/`tag_2`/`tag_3` leakage trap

These columns are populated for ~95%+ of rows and correlate strongly
with `queue` (e.g. `tag_1='Billing'` almost always implies the Billing
queue). It's tempting to fold them into the text features — **don't**.
They are dataset annotations added *after* a human triaged the ticket. A
real incoming email never arrives with tags attached; using them as
model input would inflate metrics in a way that's meaningless in
production (classic data leakage). We verify the correlation below, then
explicitly exclude these columns from every model input.

In [9]:
print("tag_1 top values:")
print(df_en['tag_1'].value_counts().head(10))
print()

# spot-check correlation that makes this tempting (and dangerous)
sample = df_en[df_en['tag_1'].isin(['Billing', 'Security'])][['tag_1', 'queue']].head(10)
sample

tag_1 top values:
tag_1
Technical      3057
Security       2033
Bug            1253
Feedback       1120
Feature         867
Billing         727
Performance     504
Customer        417
Crash           330
Outage          302
Name: count, dtype: int64



,tag_1,queue
4,Security,Customer Service
5,Security,Technical Support
14,Security,Technical Support
19,Security,Product Support
22,Security,Billing and Payments
26,Security,Customer Service
30,Billing,Billing and Payments
33,Security,Technical Support
43,Security,Billing and Payments
44,Security,Customer Service


**Decision: `tag_1`/`tag_2`/`tag_3` are excluded from all model imput, for every model trained in this project.**They may optionally be
used later for post-hoc validation (e.g. "does our predicted category
usually agree with tag_1?"), never as a training feature.

5. Placeholder token audit

In [10]:
tokens = set()
for b in df_en['body'].dropna():
    for m in re.findall(r'<[a-zA-Z_]+>', b):
        tokens.add(m)
print("Placeholder tokens found in the synthetic text:", tokens)

Placeholder tokens found in the synthetic text: {'<website_url>', '<br>', '<tel_num>', '<name>', '<user>', '<email>', '<acc_num>', '<ref_num>'}


All variants are covered by one generic regex (`<[^>]+>`), applied in cleaning step below.

6. Missing subject

In [11]:
missing_subject = df_en['subject'].isna().sum()
print(f"Rows with missing subject: {missing_subject} of {len(df_en)}")
df_en[df_en['subject'].isna()][['body']].head(2)

Rows with missing subject: 1032 of 11923


,body
19,"To Whom It May Concern, I am contacting you to report a possible data breach, which may be related to outdated softw..."
42,"During the last deployment, users encountered notable performance issues with the application. This might be due to ..."


We track it explicitly with a `subject_missing` flag so it
can inform downstream review logic (a prediction made without a subject
is working from less signal than usual).

7. Near-empty bodies

In [12]:
short = df_en[df_en['body'].str.len() < 20]
print(f"Rows with body < 20 chars: {len(short)} of {len(df_en)}")
short[['subject', 'body', 'queue']]

Rows with body < 20 chars: 25 of 11923


,subject,body,queue
485,NaN,Hello support team,Customer Service
549,NaN,Help,Billing and Payments
1398,Challenge with System Efficiency,Seeking assistance,Returns and Exchanges
2506,Problem with Data Synchronization Process,Resolve the problem,Product Support
2651,Customer Support for Brand Growth,Can you assist?,Product Support
3094,Software Assistance Required,There was a crash,Technical Support
3582,Missing Investment Information,Support Request,Customer Service
4145,NaN,Support needed,Billing and Payments
5022,Encountering Periodic Performance Deterioration,Resolution Required,Customer Service
5059,Boost Investment in Data Analytics Tools Optimization,Assistance Required,Technical Support


In [13]:
print("Class distribution of these near-empty rows:")
print(short['queue'].value_counts())

Class distribution of these near-empty rows:
queue
Customer Service                   5
Product Support                    5
Billing and Payments               3
Returns and Exchanges              3
IT Support                         3
Technical Support                  2
Service Outages and Maintenance    2
Sales and Pre-Sales                1
General Inquiry                    1
Name: count, dtype: int64


Only 25 rows, spread thinly across every category (max 5 in any one
class) — dropping them from **training** doesn't starve any class.

Important distinction: this only affects which *historical* rows we
train on. A real *incoming* email this short at inference time is still
handled — flagged via the `low_information_body` engineered feature
(Section 9), not silently dropped or rejected.

8. Category taxonomy mapping

Reflects final decision from error-analysis investigation: `Product Support`, `Account` and `General Inquiry` were merged after manual inspection showed their ground-truth labels were frequently inconsistent in this synthetic dataset. `Billing`, `Technical Support`, and `Sales` were textually
distinct and kept separate.

In [14]:
df_en['queue'].value_counts()

queue
Technical Support                  3412
Product Support                    2232
Customer Service                   1859
IT Support                         1391
Billing and Payments               1302
Returns and Exchanges               582
Service Outages and Maintenance     442
Sales and Pre-Sales                 330
Human Resources                     205
General Inquiry                     168
Name: count, dtype: int64

In [15]:
QUEUE_TO_CATEGORY = {
    "Technical Support": "Technical Support",
    "IT Support": "Technical Support",
    "Service Outages and Maintenance": "Technical Support",
    "Billing and Payments": "Billing",
    "Sales and Pre-Sales": "Sales",
    "Product Support": "General Inquiry",
    "Customer Service": "General Inquiry",
    "General Inquiry": "General Inquiry",
    "Returns and Exchanges": "General Inquiry",
    # "Human Resources" intentionally dropped: internal HR tickets are out
    # of scope for a customer-facing router, and the class is tiny (205 rows).
}

CATEGORY_TO_DEPARTMENT = {
    "Billing": "Finance",
    "Technical Support": "Technical Support Team",
    "Sales": "Sales",
    "General Inquiry": "General Support",
}

MIN_BODY_LENGTH_FOR_TRAINING = 20

9.  Apply cleaning pipeline

In [16]:
def clean_text(text: str) -> str:
    # Strips generator placeholder tokens & normalizes whitespace.
    # Keeps punctuation since TF-IDF bigrams benefit from phrases like 'not working'
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [17]:
df_clean = df_en.dropna(subset=['queue', 'priority']).copy()
df_clean['category'] = df_clean['queue'].map(QUEUE_TO_CATEGORY)
dropped = df_clean['category'].isna().sum()
print(f"Dropping {dropped} rows with unmapped queue (Human Resources)")
df_clean = df_clean.dropna(subset=['category'])

df_clean['subject_missing'] = df_clean['subject'].isna().astype(int)
df_clean['subject'] = df_clean['subject'].apply(clean_text)
df_clean['body'] = df_clean['body'].apply(clean_text)

before = len(df_clean)
df_clean = df_clean[df_clean['body'].str.len() >= MIN_BODY_LENGTH_FOR_TRAINING]
print(f"Dropped {before - len(df_clean)} near-empty-body rows from training set")

df_clean['text'] = (df_clean['subject'] + ' ' + df_clean['subject'] + ' ' + df_clean['body']).str.strip()
df_clean['priority'] = df_clean['priority'].str.upper()
df_clean['department'] = df_clean['category'].map(CATEGORY_TO_DEPARTMENT)

print(f"Final row count: {len(df_clean)}")

Dropping 205 rows with unmapped queue (Human Resources)


Dropped 27 near-empty-body rows from training set
Final row count: 11691


In [18]:
df_clean.shape

(11691, 19)

In [19]:
df_clean['queue'].value_counts()

queue
Technical Support                  3410
Product Support                    2226
Customer Service                   1854
IT Support                         1388
Billing and Payments               1298
Returns and Exchanges               579
Service Outages and Maintenance     440
Sales and Pre-Sales                 329
General Inquiry                     167
Name: count, dtype: int64

In [20]:
df_clean['category'].value_counts()

category
Technical Support    5238
General Inquiry      4826
Billing              1298
Sales                 329
Name: count, dtype: int64

10. Engineering numeric features

In [21]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11691 entries, 1 to 19997
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   subject          11691 non-null  object
 1   body             11691 non-null  object
 2   answer           11688 non-null  object
 3   type             11691 non-null  object
 4   queue            11691 non-null  object
 5   priority         11691 non-null  object
 6   language         11691 non-null  object
 7   tag_1            11691 non-null  object
 8   tag_2            11681 non-null  object
 9   tag_3            11649 non-null  object
 10  tag_4            10849 non-null  object
 11  tag_5            7761 non-null   object
 12  tag_6            4482 non-null   object
 13  tag_7            2534 non-null   object
 14  tag_8            1311 non-null   object
 15  category         11691 non-null  object
 16  subject_missing  11691 non-null  int32 
 17  text             11691 non-null  obj

In [22]:
feature_rows = [extract_numeric_features(s, b) for s, b in zip(df_clean['subject'], df_clean['body'])]
feature_df = pd.DataFrame(feature_rows, index=df_clean.index)
df_clean = pd.concat([df_clean, feature_df], axis=1)

df_clean[FEATURE_NAMES].describe()

,char_count,word_count,exclamation_count,all_caps_ratio,has_question_mark,urgency_keyword_count,missing_subject,low_information_body
count,11691.000000,11691.000000,11691.000000,11691.000000,11691.000000,11691.000000,11691.000000,11691.0
mean,417.596014,62.975024,0.005816,0.005890,0.343512,0.243777,0.086049,0.0
std,230.443271,36.433711,0.076047,0.020237,0.474901,0.561011,0.280449,0.0
min,29.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
25%,234.000000,34.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
50%,390.000000,58.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
75%,572.000000,87.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.0
max,1825.000000,276.000000,1.000000,0.500000,1.000000,4.000000,1.000000,0.0


Do there features actually separate classes? 



In [23]:
df_clean.groupby('priority')[['urgency_keyword_count', 'exclamation_count', 'all_caps_ratio']].mean()

,urgency_keyword_count,exclamation_count,all_caps_ratio
priority,,,
HIGH,0.341474,0.002640,0.006008
LOW,0.153846,0.006953,0.005540
MEDIUM,0.194840,0.008256,0.005944


11. Final sanity checks

In [24]:
print("Final category distribution:")
print(df_clean['category'].value_counts())
print()
print("Final priority distribution:")
print(df_clean['priority'].value_counts())
print()
assert df_clean['text'].isna().sum() == 0, "Found null text rows"
assert df_clean['category'].isna().sum() == 0, "Found unmapped categories"
assert df_clean[FEATURE_NAMES].isna().sum().sum() == 0, "Found null engineered features"
print("Sanity checks passed.")

Final category distribution:
category
Technical Support    5238
General Inquiry      4826
Billing              1298
Sales                 329
Name: count, dtype: int64

Final priority distribution:
priority
MEDIUM    4845
HIGH      4545
LOW       2301
Name: count, dtype: int64

Sanity checks passed.


In [25]:
df_clean.isna().sum()

subject                      0
body                         0
answer                       3
type                         0
queue                        0
priority                     0
language                     0
tag_1                        0
tag_2                       10
tag_3                       42
tag_4                      842
tag_5                     3930
tag_6                     7209
tag_7                     9157
tag_8                    10380
category                     0
subject_missing              0
text                         0
department                   0
char_count                   0
word_count                   0
exclamation_count            0
all_caps_ratio               0
has_question_mark            0
urgency_keyword_count        0
missing_subject              0
low_information_body         0
dtype: int64

In [26]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11691 entries, 1 to 19997
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   subject                11691 non-null  object 
 1   body                   11691 non-null  object 
 2   answer                 11688 non-null  object 
 3   type                   11691 non-null  object 
 4   queue                  11691 non-null  object 
 5   priority               11691 non-null  object 
 6   language               11691 non-null  object 
 7   tag_1                  11691 non-null  object 
 8   tag_2                  11681 non-null  object 
 9   tag_3                  11649 non-null  object 
 10  tag_4                  10849 non-null  object 
 11  tag_5                  7761 non-null   object 
 12  tag_6                  4482 non-null   object 
 13  tag_7                  2534 non-null   object 
 14  tag_8                  1311 non-null   object 
 15  categor

In [27]:
print(type(df_clean['subject']))

<class 'pandas.core.series.Series'>


In [28]:
keep_cols = ['subject', 'body', 'text', 'category', 'priority', 'type', 'department'] + FEATURE_NAMES
final_df = df_clean[keep_cols].reset_index(drop=True)
final_df.to_csv(CLEAN_PATH, index=False)
print(f"Saved {len(final_df)} rows x {len(final_df.columns)} columns to {CLEAN_PATH}")
final_df.head(3)

Saved 11691 rows x 15 columns to ..\data\clean_tickets.csv


,subject,body,text,category,priority,type,department,char_count,word_count,exclamation_count,all_caps_ratio,has_question_mark,urgency_keyword_count,missing_subject,low_information_body
0,Customer Support Inquiry,Seeking information on digital strategies that can aid in brand growth and details on the available services. Lookin...,Customer Support Inquiry Customer Support Inquiry Seeking information on digital strategies that can aid in brand gr...,General Inquiry,MEDIUM,Request,General Support,250,41,0,0.0000,0,0,0,0
1,Data Analytics for Investment,I am contacting you to request information on data analytics tools that can be utilized with the Eclipse IDE for enh...,Data Analytics for Investment Data Analytics for Investment I am contacting you to request information on data analy...,General Inquiry,MEDIUM,Request,General Support,726,109,0,0.0333,0,0,0,0
2,Security,"Dear Customer Support, I am reaching out to inquire about the security protocols you have in place to protect medica...","Security Security Dear Customer Support, I am reaching out to inquire about the security protocols you have in place...",General Inquiry,MEDIUM,Request,General Support,684,114,0,0.0116,1,0,0,0


In [29]:
reloaded = pd.read_csv(CLEAN_PATH)
nan_subjects_on_reload = reloaded['subject'].isna().sum()
print(f"subject NaN count immediately after CSV round-trip: {nan_subjects_on_reload}")
print("(Expected > 0 here — this confirms the bug this cell is documenting.)")
print(" Every downstream consumer applies .fillna('') on load, so this")
print(" is safe as long as that guard stays in place — see training/*.py.)")

# Sanity-check fix pattern downstream code relies on:
reloaded['subject'] = reloaded['subject'].fillna('')
assert reloaded['subject'].isna().sum() == 0
print("\nAfter .fillna(''): 0 NaN subjects — fix confirmed working.")

subject NaN count immediately after CSV round-trip: 1006
(Expected > 0 here — this confirms the bug this cell is documenting.)
 Every downstream consumer applies .fillna('') on load, so this
 is safe as long as that guard stays in place — see training/*.py.)

After .fillna(''): 0 NaN subjects — fix confirmed working.


In [30]:
reloaded['subject'].isna().sum()

0

In [31]:
reloaded.isna().sum()

subject                  0
body                     0
text                     0
category                 0
priority                 0
type                     0
department               0
char_count               0
word_count               0
exclamation_count        0
all_caps_ratio           0
has_question_mark        0
urgency_keyword_count    0
missing_subject          0
low_information_body     0
dtype: int64